<a href="https://colab.research.google.com/github/k-nikhil-coder/100-day-ai-engineer/blob/main/day21_model_drift_monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy scipy

In [2]:
"""
Day 21: Statistical Drift & Data Quality Monitor
Runs baseline Kolmogorov-Smirnov (KS) tests to detect feature distribution shift
between training baselines and incoming inference payloads.
"""

from typing import Dict, Any
import numpy as np
import pandas as pd
from scipy import stats


def generate_synthetic_data():
    """Generates a reference training distribution and two production inference batches."""
    np.random.seed(42)
    n_samples = 1000

    # Baseline reference data (e.g., California Housing income feature)
    ref_income = np.random.normal(loc=3.87, scale=1.89, size=n_samples)
    ref_house_age = np.random.uniform(low=1.0, high=52.0, size=n_samples)

    df_reference = pd.DataFrame({
        "med_inc": ref_income,
        "house_age": ref_house_age
    })

    # Batch 1: In-distribution incoming traffic (No drift)
    batch_normal = pd.DataFrame({
        "med_inc": np.random.normal(loc=3.85, scale=1.90, size=200),
        "house_age": np.random.uniform(low=1.0, high=52.0, size=200)
    })

    # Batch 2: Out-of-distribution traffic (Covariate shift / Inflation surge)
    batch_drifted = pd.DataFrame({
        "med_inc": np.random.normal(loc=6.20, scale=2.10, size=200),  # Mean shifted significantly
        "house_age": np.random.uniform(low=1.0, high=52.0, size=200)
    })

    return df_reference, batch_normal, batch_drifted


def detect_feature_drift(
    reference: pd.Series,
    current: pd.Series,
    alpha: float = 0.05
) -> Dict[str, Any]:
    """
    Performs a Two-Sample Kolmogorov-Smirnov (KS) test.
    - Null Hypothesis (H0): Both distributions are identical.
    - If p-value < alpha: Reject H0 -> Statistically significant drift detected!
    """
    ks_stat, p_value = stats.ks_2samp(reference, current)
    is_drifted = p_value < alpha

    return {
        "ks_statistic": round(float(ks_stat), 4),
        "p_value": round(float(p_value), 6),
        "drift_detected": bool(is_drifted),
        "ref_mean": round(float(reference.mean()), 3),
        "current_mean": round(float(current.mean()), 3)
    }


def run_monitoring_suite(reference_df: pd.DataFrame, incoming_df: pd.DataFrame, batch_name: str):
    print(f"\n================ MONITORING REPORT: {batch_name} ================")

    for column in reference_df.columns:
        result = detect_feature_drift(reference_df[column], incoming_df[column])

        status = "🚨 ALERT: DRIFT DETECTED" if result["drift_detected"] else "✅ HEALTHY (NO DRIFT)"
        print(f"\nFeature: [{column}] -> {status}")
        print(f"  • Baseline Mean: {result['ref_mean']} | Current Mean: {result['current_mean']}")
        print(f"  • KS Statistic:  {result['ks_statistic']} | p-value: {result['p_value']}")


if __name__ == "__main__":
    ref_data, clean_batch, drifted_batch = generate_synthetic_data()

    # Run check on clean batch
    run_monitoring_suite(ref_data, clean_batch, "Production Batch #101 (Normal Traffic)")

    # Run check on drifted batch
    run_monitoring_suite(ref_data, drifted_batch, "Production Batch #102 (High Inflation Shift)")


================ MONITORING REPORT: Production Batch #101 (Normal Traffic) ================

Feature: [med_inc] -> ✅ HEALTHY (NO DRIFT)
  • Baseline Mean: 3.907 | Current Mean: 3.835
  • KS Statistic:  0.038 | p-value: 0.964914

Feature: [house_age] -> ✅ HEALTHY (NO DRIFT)
  • Baseline Mean: 26.686 | Current Mean: 25.181
  • KS Statistic:  0.079 | p-value: 0.240858

================ MONITORING REPORT: Production Batch #102 (High Inflation Shift) ================

Feature: [med_inc] -> 🚨 ALERT: DRIFT DETECTED
  • Baseline Mean: 3.907 | Current Mean: 6.325
  • KS Statistic:  0.491 | p-value: 0.0

Feature: [house_age] -> ✅ HEALTHY (NO DRIFT)
  • Baseline Mean: 26.686 | Current Mean: 25.737
  • KS Statistic:  0.068 | p-value: 0.41211
